# The Vandermonde matrix

The design matrix of a polynomial regression is a Vandermonde matrix: row $i$ contains $1, x_i, x_i^2, \ldots, x_i^d$. Its Gram matrix $X^{\mathsf T}X$ is filled with power sums. On a symmetric set of inputs, every odd power sum is zero, and the even and odd coefficients separate.


## Five symmetric inputs

Take $x = -2, -1, 0, 1, 2$ and degree $2$. Then

$$
X = \begin{bmatrix}
1 & -2 & 4 \\
1 & -1 & 1 \\
1 & 0 & 0 \\
1 & 1 & 1 \\
1 & 2 & 4
\end{bmatrix}.
$$

The power sums that enter $X^{\mathsf T}X$ are

$$
\begin{aligned}
\sum 1 &= 5, \\
\sum x &= 0, \\
\sum x^2 &= 4 + 1 + 0 + 1 + 4 = 10, \\
\sum x^3 &= -8 - 1 + 0 + 1 + 8 = 0, \\
\sum x^4 &= 16 + 1 + 0 + 1 + 16 = 34.
\end{aligned}
$$

Therefore

$$
X^{\mathsf T}X = \begin{bmatrix}
5 & 0 & 10 \\
0 & 10 & 0 \\
10 & 0 & 34
\end{bmatrix}.
$$

The middle row and column are zero off the diagonal because $\sum x = 0$ and $\sum x^3 = 0$. Even columns are orthogonal to odd columns. The slope has its own one-by-one block, and the intercept shares a two-by-two block with the quadratic coefficient:

$$
\begin{bmatrix} 5 & 10 \\ 10 & 34 \end{bmatrix}, \qquad
\det = 5\cdot 34 - 10\cdot 10 = 70.
$$

The determinant of the whole Gram matrix is $10 \cdot 70 = 700 \neq 0$, so these three columns are linearly independent. Distinct inputs and a degree below the sample size are what produce that independence: a nonzero polynomial of degree $d$ has at most $d$ roots, and it cannot vanish at $d + 1$ distinct sample points.


In [1]:
# Power sums, the Gram matrix, and its determinant.
from fractions import Fraction
import numpy as np
x = [-2, -1, 0, 1, 2]
X = np.array([[1, xi, xi ** 2] for xi in x], dtype=object)
gram = X.T @ X
even = np.array([[5, 10], [10, 34]], dtype=object)
det_even = int(even[0, 0]) * int(even[1, 1]) - int(even[0, 1]) ** 2
print("X'X")
print(gram)
print("even-block determinant", det_even)
assert list(gram[0]) == [5, 0, 10]
assert list(gram[1]) == [0, 10, 0]
assert list(gram[2]) == [10, 0, 34]
assert det_even == 70
assert sum(xi ** 3 for xi in x) == 0


X'X
[[5 0 10]
 [0 10 0]
 [10 0 34]]
even-block determinant 70


## One change: drop the center point

Remove $x = 0$. The remaining inputs $-2, -1, 1, 2$ are still symmetric, so the odd power sums remain zero. The diagonal entry that was $5$ becomes $4$, and

$$
\sum x^2 = 10, \qquad \sum x^4 = 34
$$

are unchanged because $0^2$ and $0^4$ contributed nothing. The even-block determinant becomes $4\cdot 34 - 100 = 36$. Deleting the center observation does not create a dependence, and it does not destroy the separation of even and odd coefficients.


In [2]:
# Without x = 0 the odd sums are still zero and the even block is still invertible.
x = [-2, -1, 1, 2]
sum_x3 = sum(xi ** 3 for xi in x)
even_det = 4 * 34 - 10 * 10
print("sum of cubes", sum_x3)
print("even determinant", even_det)
assert sum_x3 == 0
assert even_det == 36


sum of cubes 0
even determinant 36


## Pitfall

Repeating an input does not add a new power of $x$. Two copies of $x = 1$ produce two identical rows in a polynomial design. The Gram matrix can remain invertible, because the distinct values may still span the polynomial space, but the rank cannot exceed one plus the degree, and it also cannot exceed the number of distinct inputs. Five copies of a single $x$ give a rank-$1$ polynomial design no matter how many powers are appended: every column is a scalar multiple of the column of ones.


In [3]:
# Repeated inputs do not raise the rank above the number of distinct values.
import numpy as np
x = np.array([1.0, 1, 1, 1, 1])
X = np.column_stack([x ** k for k in range(3)])
print("design")
print(X)
print("rank", np.linalg.matrix_rank(X))
assert np.linalg.matrix_rank(X) == 1


design
[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]
rank 1


## Summary

- The degree-$2$ design on $x = -2, -1, 0, 1, 2$ has Gram matrix $\operatorname{diag}$-blocked into the slope $10$ and the even block with determinant $70$.
- Odd power sums vanish because the inputs are symmetric about zero. Even and odd coefficients do not share equations.
- Distinct inputs, not repeated rows, are what give a polynomial column its rank.
